# FM回归
将AED与fm_reg_controls表进合并，用于进行回归  

## 导入库

In [5]:
import polars as pl
import dotenv
import os

dotenv.load_dotenv()

True

## 超参数

In [6]:
TASK_PREFIX = 'baseline1'
EMPIRICAL_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_PREFIX}'
BASE_LINE_REG_DIR = EMPIRICAL_DIR + '/baseline_reg'

DATABASE_URL = os.getenv('POSTGRES_URL')

## 读取数据
- 从数据库中读取控制变量  
- 从BASE_LINE_REG_DIR中获取aed.parquet 

### 读数据库 

In [7]:
# 使用polars从数据库读取fm_reg_controls表数据，指定schema避免类型推断问题
fm_reg_controls_schema = {
    'stkcd': pl.Utf8,              # 股票代码
    'accper': pl.Date,             # 会计期间
    'betavals': pl.Float64,        # 贝塔值
    'bm_ratio': pl.Float64,        # 账面市值比
    'gross_margin': pl.Float64,    # 毛利率
    'investment_ratio': pl.Float64, # 投资比率
    'ln_market_value': pl.Float64   # 对数市值
}

fm_reg_controls_df = pl.read_database_uri(
    query="SELECT * FROM statics.fm_reg_controls",
    uri=DATABASE_URL,
    schema_overrides=fm_reg_controls_schema
)

print(f"数据类型检查:")
print(fm_reg_controls_df.dtypes)
fm_reg_controls_df.head()

数据类型检查:
[String, Date, Float64, Float64, Float64, Float64, Float64]


stkcd,accper,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,date,f64,f64,f64,f64,f64
"""000001""",1997-01-01,0.93259,null,null,null,16.435539
"""000001""",1997-02-01,0.88586,null,null,null,16.433457
"""000001""",1997-03-01,1.06864,null,null,null,16.785434
"""000001""",1997-04-01,1.01343,null,null,null,17.082685
"""000001""",1997-05-01,1.22579,null,null,null,16.994263


### 读取AED因子

In [9]:
aed = pl.read_parquet(BASE_LINE_REG_DIR + '/基准回归-AED因子.parquet')
aed.head()

date,portfolio,AED,return
date,str,f64,f64
2021-03-01,"""002483""",2.596032,0.0485
2018-07-01,"""300226""",1.341226,-0.1245
2021-03-01,"""600502""",1.824548,-0.0382
2020-10-01,"""002628""",2.59455,0.0077
2015-02-01,"""300251""",1.082017,0.0035


## 处理数据 
对于Controls数据，由于部分数据是季频，用该季数据填充该月数据  

In [ ]:
### 季度数据填充

# 定义季度数据字段
quarterly_columns = ['bm_ratio', 'gross_margin', 'investment_ratio']

def expand_quarterly_to_monthly(df, quarterly_cols, date_col='accper', stkcd_col='stkcd'):
    """
    将季度数据扩展到月度，使用季度末数据填充该季度其他月份
    使用Polars原生操作避免Object类型问题
    """
    # 确保日期列是datetime类型并提取月份
    df = df.with_columns([
        pl.col(date_col).cast(pl.Date),
        pl.col(date_col).dt.month().alias('month')
    ])

    # 过滤出季度末数据
    quarter_end_data = df.filter(pl.col('month').is_in([3, 6, 9, 12]))

    # 为每个季度创建扩展数据
    expanded_dfs = []

    for quarter_end_month in [3, 6, 9, 12]:
        qtr_data = quarter_end_data.filter(pl.col('month') == quarter_end_month)

        # 确定该季度包含的月份
        if quarter_end_month == 3:      # Q1: 1,2,3月
            months = [1, 2, 3]
        elif quarter_end_month == 6:    # Q2: 4,5,6月
            months = [4, 5, 6]
        elif quarter_end_month == 9:    # Q3: 7,8,9月
            months = [7, 8, 9]
        else:  # 12 - Q4: 10,11,12月
            months = [10, 11, 12]

        # 为每个月创建数据
        for month in months:
            month_data = qtr_data.with_columns(
                pl.date(pl.col(date_col).dt.year(), pl.lit(month), pl.lit(1)).alias(date_col)
            )
            expanded_dfs.append(month_data)

    # 合并所有扩展数据
    if expanded_dfs:
        expanded_df = pl.concat(expanded_dfs)
        # 按股票代码和日期排序
        expanded_df = expanded_df.sort([stkcd_col, date_col])
        # 移除辅助列
        expanded_df = expanded_df.drop('month')
        return expanded_df
    else:
        return pl.DataFrame()

# 应用季度数据填充
fm_reg_controls_monthly_df = expand_quarterly_to_monthly(
    fm_reg_controls_df,
    quarterly_columns
)

print("\n填充后数据预览:")
fm_reg_controls_monthly_df.head(10)

# 注意：后续分析请使用填充后的数据 fm_reg_controls_monthly_df
# 它包含了扩展到月度的季度财务数据

开始季度数据填充...
填充后数据形状: (1005867, 7)
原始数据行数: 1000000
填充后数据行数: 1005867
数据量增加倍数: 1.01倍

填充后数据预览:


stkcd,accper,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,date,f64,f64,f64,f64,f64
"""000001""",1997-01-01,1.06864,null,null,null,16.785434
"""000001""",1997-02-01,1.06864,null,null,null,16.785434
"""000001""",1997-03-01,1.06864,null,null,null,16.785434
"""000001""",1997-04-01,1.06346,0.550942,null,0.978269,17.109268
"""000001""",1997-05-01,1.06346,0.550942,null,0.978269,17.109268
"""000001""",1997-06-01,1.06346,0.550942,null,0.978269,17.109268
"""000001""",1997-07-01,0.85352,null,null,null,16.943189
"""000001""",1997-08-01,0.85352,null,null,null,16.943189
"""000001""",1997-09-01,0.85352,null,null,null,16.943189


In [11]:
fm_reg_controls_df.filter(pl.col('gross_margin').is_not_null())

stkcd,accper,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,date,f64,f64,f64,f64,f64
